# Fairness check

Take the sbert scores. For each resume + job, compare the original score with the score after we changed the name / pronouns / university.

In [ ]:
import os, pandas as pd
os.makedirs("results", exist_ok=True)

In [ ]:
# upload sbert_scores.csv
from google.colab import files
files.upload()

In [ ]:
df = pd.read_csv("sbert_scores.csv")
df.head()

In [ ]:
orig = df[df.version == "original"][["resume_id", "job_id", "job_title", "similarity_score"]]
orig = orig.rename(columns={"similarity_score": "original_score"})

changed = df[df.version != "original"].rename(columns={"similarity_score": "changed_score"})

cmp = changed.merge(orig, on=["resume_id", "job_id", "job_title"], how="left")
cmp["score_difference"] = cmp["changed_score"] - cmp["original_score"]
cmp["absolute_difference"] = cmp["score_difference"].abs()
cmp.head()

In [ ]:
summary = cmp.groupby("changed_signal").agg(
    average_score_difference=("score_difference", "mean"),
    average_absolute_difference=("absolute_difference", "mean"),
    max_absolute_difference=("absolute_difference", "max"),
    min_score_difference=("score_difference", "min"),
    max_score_difference=("score_difference", "max"),
).reset_index()
summary

Name change moves the score the most on average. Pronoun the least. University in between.

In [ ]:
cmp.to_csv("results/fairness_comparison.csv", index=False)
summary.to_csv("results/fairness_summary.csv", index=False)

In [ ]:
from google.colab import files
files.download("results/fairness_comparison.csv")
files.download("results/fairness_summary.csv")